# E-commerce Profitability & Revenue Leakage Analysis

## Section 1 - Project Overview

**Business problem statement:** This project analyzes where an e-commerce marketplace may lose revenue through cancellation loss, high freight burden, low estimated profitability, delayed delivery, and weak seller performance.

**Dataset description:** The Olist Brazilian E-commerce dataset contains 100,000+ real marketplace orders from 2016-2018 across orders, order items, products, sellers, customers, reviews, payments, and category translation tables.

**Tools used:** Python, pandas, numpy, matplotlib, seaborn, MySQL, and Power BI.

**Important assumption:** Olist does not provide actual product cost. Estimated cost is assumed to be 60% of item price. Estimated profit is calculated as `price - estimated_cost - freight_value`.

## Section 2 - Data Loading & Exploration

This section loads all eight Olist CSV files, checks the shape and data types of each table, and builds a missing-value summary so data quality issues are visible before analysis.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = '../data/raw/'

orders = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
reviews = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
payments = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
translation = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'sellers': sellers,
    'customers': customers,
    'reviews': reviews,
    'payments': payments,
    'translation': translation
}

for name, df in tables.items():
    print(f'\n{name.upper()}')
    print('Shape:', df.shape)
    print(df.dtypes)

missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in tables.items()],
    axis=1
).fillna(0).astype(int)
missing_summary

**Table meaning:** Orders stores status and dates; order items stores item price and real freight; products stores product attributes; sellers stores seller location; customers stores customer location; reviews stores customer scores; payments stores payment method and value; translation maps Portuguese categories to English.

## Section 3 - Data Cleaning

This section converts date fields, separates delivered orders from canceled/unavailable orders, removes missing delivery dates from the main analysis, and enriches item-level data with English product category names.

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

canceled_df = orders[orders['order_status'].isin(['canceled', 'unavailable'])].copy()
delivered_df = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].notna())
].copy()

items_enriched = (
    order_items
    .merge(products[['product_id', 'product_category_name', 'product_weight_g']], on='product_id', how='left')
    .merge(translation, on='product_category_name', how='left')
)
items_enriched['product_category_name_english'] = items_enriched['product_category_name_english'].fillna('unknown')

analysis_df = (
    items_enriched
    .merge(delivered_df, on='order_id', how='inner')
    .merge(customers[['customer_id', 'customer_city', 'customer_state']], on='customer_id', how='left')
    .merge(sellers[['seller_id', 'seller_city', 'seller_state']], on='seller_id', how='left')
)

canceled_items_df = (
    items_enriched
    .merge(canceled_df[['order_id', 'order_status']], on='order_id', how='inner')
)

print('Delivered analysis rows:', analysis_df.shape)
print('Canceled/unavailable item rows:', canceled_items_df.shape)

**Cleaning notes:** Date conversion enables time-series and delay calculations. Delivered orders with missing delivery dates are removed because delivery delay analysis requires actual and estimated delivery dates. Canceled and unavailable orders are stored separately as cancellation loss, not return loss. Category translation is added so charts are readable for business users.

## Section 4 - Exploratory Data Analysis

This section shows order status mix, payment behavior, monthly order volume, and top product categories by order count to understand marketplace activity before profitability analysis.

In [ ]:
def annotate_bars(ax, fmt='{:.0f}'):
    for container in ax.containers:
        labels = []
        for value in container.datavalues:
            labels.append(fmt.format(value))
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)

status_counts = orders['order_status'].value_counts()
ax = sns.barplot(x=status_counts.index, y=status_counts.values, palette='viridis')
ax.set_title('Order Status Distribution Across Olist Orders')
ax.set_xlabel('Order Status')
ax.set_ylabel('Number of Orders')
plt.xticks(rotation=45)
annotate_bars(ax)
plt.tight_layout()
plt.show()

payment_counts = payments['payment_type'].value_counts()
ax = sns.barplot(x=payment_counts.index, y=payment_counts.values, palette='mako')
ax.set_title('Payment Method Distribution by Transaction Count')
ax.set_xlabel('Payment Method')
ax.set_ylabel('Number of Payment Records')
annotate_bars(ax)
plt.tight_layout()
plt.show()

monthly_orders = delivered_df.set_index('order_purchase_timestamp').resample('M')['order_id'].nunique()
ax = monthly_orders.plot(marker='o', color='#2A6FBB')
ax.set_title('Monthly Delivered Order Volume Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Delivered Orders')
for x, y in zip(monthly_orders.index, monthly_orders.values):
    ax.annotate(f'{int(y)}', (x, y), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

top_category_orders = analysis_df.groupby('product_category_name_english')['order_id'].nunique().sort_values(ascending=False).head(10)
ax = sns.barplot(x=top_category_orders.values, y=top_category_orders.index, palette='crest')
ax.set_title('Top 10 Product Categories by Delivered Order Count')
ax.set_xlabel('Delivered Orders')
ax.set_ylabel('Product Category')
annotate_bars(ax)
plt.tight_layout()
plt.show()

**Observation placeholders:**

- Order status chart: delivered orders are expected to dominate; canceled/unavailable orders should be reviewed as cancellation loss.
- Payment chart: the leading payment type shows the strongest customer payment preference; smaller methods may need checkout monitoring.
- Monthly trend: growth and seasonality should be checked around peak months.
- Category chart: top order-count categories may not always be the most profitable categories.

## Section 5 - Revenue & Profitability Analysis

This section estimates item-level cost and profit using the portfolio assumption that estimated cost equals 60% of item price, then identifies categories to grow or investigate.

In [ ]:
analysis_df['estimated_cost'] = analysis_df['price'] * 0.60
analysis_df['estimated_profit'] = analysis_df['price'] - analysis_df['estimated_cost'] - analysis_df['freight_value']
analysis_df['profit_margin'] = np.where(analysis_df['price'] > 0, analysis_df['estimated_profit'] / analysis_df['price'] * 100, np.nan)

category_profit = analysis_df.groupby('product_category_name_english').agg(
    total_revenue=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    estimated_profit=('estimated_profit', 'sum'),
    item_count=('order_id', 'count')
).reset_index()
category_profit['profit_margin'] = category_profit['estimated_profit'] / category_profit['total_revenue'] * 100

top_revenue = category_profit.sort_values('total_revenue', ascending=False).head(10)
ax = sns.barplot(data=top_revenue, x='total_revenue', y='product_category_name_english', palette='Blues_r')
ax.set_title('Top 10 Product Categories by Delivered Revenue')
ax.set_xlabel('Total Revenue')
ax.set_ylabel('Product Category')
annotate_bars(ax, fmt='{:.0f}')
plt.tight_layout()
plt.show()

top_margin = category_profit[category_profit['item_count'] >= 20].sort_values('profit_margin', ascending=False).head(10)
ax = sns.barplot(data=top_margin, x='profit_margin', y='product_category_name_english', palette='Greens_r')
ax.set_title('Top 10 Product Categories by Estimated Profit Margin')
ax.set_xlabel('Estimated Profit Margin %')
ax.set_ylabel('Product Category')
annotate_bars(ax, fmt='{:.1f}%')
plt.tight_layout()
plt.show()

bottom_margin = category_profit[category_profit['item_count'] >= 20].sort_values('profit_margin').head(10)
ax = sns.barplot(data=bottom_margin, x='profit_margin', y='product_category_name_english', color='#C44E52')
ax.set_title('Bottom 10 Product Categories by Estimated Profit Margin')
ax.set_xlabel('Estimated Profit Margin %')
ax.set_ylabel('Product Category')
annotate_bars(ax, fmt='{:.1f}%')
plt.tight_layout()
plt.show()

ax = sns.scatterplot(data=category_profit, x='total_revenue', y='profit_margin', size='item_count', sizes=(40, 400), color='#4C72B0')
ax.set_title('Revenue vs Estimated Profit Margin by Product Category')
ax.set_xlabel('Total Revenue')
ax.set_ylabel('Estimated Profit Margin %')
for _, row in category_profit.sort_values('total_revenue', ascending=False).head(10).iterrows():
    ax.annotate(row['product_category_name_english'], (row['total_revenue'], row['profit_margin']), fontsize=8)
plt.tight_layout()
plt.show()

**Business interpretation:** Grow categories with both high revenue and strong estimated margin. Investigate categories with high revenue but weak margin because freight or pricing may be reducing profitability.

## Section 6 - Freight Trap Analysis

This section replaces discount analysis because Olist has no discount column. It uses real freight values to flag categories where high freight percentage and low estimated margin create a Freight Trap.

In [ ]:
freight_summary = category_profit.copy()
freight_summary['freight_pct'] = freight_summary['total_freight'] / freight_summary['total_revenue'] * 100
freight_summary['freight_trap_flag'] = np.select(
    [
        (freight_summary['freight_pct'] > 15) & (freight_summary['profit_margin'] < 10),
        (freight_summary['freight_pct'] < 10) & (freight_summary['profit_margin'] > 20)
    ],
    ['Freight Trap', 'Healthy'],
    default='Monitor'
)

flag_palette = {'Freight Trap': '#C44E52', 'Healthy': '#55A868', 'Monitor': '#F0C808'}
ax = sns.scatterplot(data=freight_summary, x='freight_pct', y='profit_margin', hue='freight_trap_flag', palette=flag_palette, s=90)
ax.set_title('Freight Percentage vs Estimated Profit Margin by Category')
ax.set_xlabel('Freight as % of Revenue')
ax.set_ylabel('Estimated Profit Margin %')
top_traps = freight_summary[freight_summary['freight_trap_flag'] == 'Freight Trap'].sort_values('freight_pct', ascending=False).head(5)
for _, row in top_traps.iterrows():
    ax.annotate(row['product_category_name_english'], (row['freight_pct'], row['profit_margin']), fontsize=8)
plt.tight_layout()
plt.show()

freight_summary[freight_summary['freight_trap_flag'] == 'Freight Trap'].sort_values('freight_pct', ascending=False)

**Recommendation placeholder:** Freight Trap categories should be reviewed for packaging changes, freight renegotiation, minimum order thresholds, or price adjustment because real freight cost is consuming estimated margin.

## Section 7 - Delivery Delay Analysis

This section measures whether delivered orders arrived early, on time, or late, then identifies customer states with the highest average delay.

In [ ]:
analysis_df['delay_days'] = (analysis_df['order_delivered_customer_date'] - analysis_df['order_estimated_delivery_date']).dt.days
order_delay = analysis_df.groupby('order_id').agg(
    delay_days=('delay_days', 'max'),
    customer_state=('customer_state', 'first')
).reset_index()
order_delay['delivery_flag'] = np.where(order_delay['delay_days'] > 0, 'Delayed', 'On Time or Early')

delivery_counts = order_delay['delivery_flag'].value_counts()
plt.pie(delivery_counts.values, labels=delivery_counts.index, autopct='%1.1f%%', colors=['#55A868', '#C44E52'])
plt.title('Overall On-Time or Early vs Delayed Delivered Orders')
plt.tight_layout()
plt.show()

ax = sns.histplot(order_delay['delay_days'], bins=40, color='#4C72B0')
ax.set_title('Distribution of Delivery Delay Days')
ax.set_xlabel('Delay Days')
ax.set_ylabel('Number of Orders')
for patch in ax.patches:
    height = patch.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', (patch.get_x() + patch.get_width() / 2, height), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

state_delay = order_delay.groupby('customer_state').agg(
    avg_delay_days=('delay_days', 'mean'),
    total_orders=('order_id', 'nunique'),
    delayed_orders=('delivery_flag', lambda x: (x == 'Delayed').sum())
).reset_index()
state_delay['delayed_pct'] = state_delay['delayed_orders'] / state_delay['total_orders'] * 100
top_state_delay = state_delay.sort_values('avg_delay_days', ascending=False).head(15)
ax = sns.barplot(data=top_state_delay, x='avg_delay_days', y='customer_state', palette='rocket')
ax.set_title('Top 15 Customer States by Average Delivery Delay')
ax.set_xlabel('Average Delay Days')
ax.set_ylabel('Customer State')
annotate_bars(ax, fmt='{:.1f}')
plt.tight_layout()
plt.show()

**Logistics interpretation:** States with the highest average delay and delayed percentage should be prioritized for carrier review, route optimization, and delivery SLA monitoring.

## Section 8 - Cancellation Loss Analysis

This section uses canceled and unavailable orders as a proxy for cancellation loss. It does not treat these records as returns because Olist has no return data.

In [ ]:
total_cancellation_loss = canceled_items_df['price'].sum()
print('Total revenue lost from canceled/unavailable orders:', round(total_cancellation_loss, 2))

delivered_category_orders = analysis_df.groupby('product_category_name_english')['order_id'].nunique()
canceled_category_orders = canceled_items_df.groupby('product_category_name_english')['order_id'].nunique()
category_cancel = pd.concat([delivered_category_orders.rename('delivered_orders'), canceled_category_orders.rename('canceled_orders')], axis=1).fillna(0)
category_cancel['cancellation_rate'] = category_cancel['canceled_orders'] / (category_cancel['delivered_orders'] + category_cancel['canceled_orders']) * 100
category_cancel = category_cancel.reset_index()

top_cancel_rate = category_cancel.sort_values('cancellation_rate', ascending=False).head(10)
ax = sns.barplot(data=top_cancel_rate, x='cancellation_rate', y='product_category_name_english', palette='Reds_r')
ax.set_title('Top Product Categories by Cancellation Rate')
ax.set_xlabel('Cancellation Rate %')
ax.set_ylabel('Product Category')
annotate_bars(ax, fmt='{:.1f}%')
plt.tight_layout()
plt.show()

top_cancel_loss = canceled_items_df.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False).head(5)
top_cancel_loss

**Business impact:** Categories with high cancellation revenue loss should be reviewed for inventory availability, seller fulfillment reliability, and order confirmation issues.

## Section 9 - Seller Performance Analysis

This section scores sellers using revenue, order count, cancellation rate, average review score, average delay days, and estimated profit contribution.

In [ ]:
seller_orders = (
    order_items[['order_id', 'seller_id', 'price', 'freight_value']]
    .merge(orders[['order_id', 'order_status', 'order_delivered_customer_date', 'order_estimated_delivery_date']], on='order_id', how='left')
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
    .merge(sellers[['seller_id', 'seller_city', 'seller_state']], on='seller_id', how='left')
)
seller_orders['estimated_profit'] = seller_orders['price'] - seller_orders['price'] * 0.60 - seller_orders['freight_value']
seller_orders['delay_days'] = (seller_orders['order_delivered_customer_date'] - seller_orders['order_estimated_delivery_date']).dt.days

seller_score = seller_orders.groupby(['seller_id', 'seller_city', 'seller_state']).agg(
    total_revenue=('price', lambda x: x[seller_orders.loc[x.index, 'order_status'] == 'delivered'].sum()),
    order_count=('order_id', 'nunique'),
    canceled_orders=('order_id', lambda x: seller_orders.loc[x.index][seller_orders.loc[x.index, 'order_status'].isin(['canceled', 'unavailable'])]['order_id'].nunique()),
    avg_review_score=('review_score', 'mean'),
    avg_delay_days=('delay_days', 'mean'),
    estimated_profit_contribution=('estimated_profit', lambda x: x[seller_orders.loc[x.index, 'order_status'] == 'delivered'].sum())
).reset_index()
seller_score['cancellation_rate'] = seller_score['canceled_orders'] / seller_score['order_count'] * 100

seller_score['performance_tier'] = np.select(
    [
        (seller_score['cancellation_rate'] > 20) | (seller_score['avg_review_score'] < 3) | (seller_score['avg_delay_days'] > 7),
        (seller_score['cancellation_rate'] < 10) & (seller_score['avg_review_score'] > 4) & (seller_score['avg_delay_days'] <= 3)
    ],
    ['Poor', 'Good'],
    default='Average'
)
seller_score['revenue_rank_pct'] = seller_score['total_revenue'].rank(pct=True)
seller_score['risk_flag'] = np.where(seller_score['revenue_rank_pct'] <= 0.10, 'At Risk', 'Normal')

tier_counts = seller_score['performance_tier'].value_counts().reindex(['Good', 'Average', 'Poor']).fillna(0)
ax = sns.barplot(x=tier_counts.index, y=tier_counts.values, palette=['#55A868', '#F0C808', '#C44E52'])
ax.set_title('Seller Tier Distribution by Performance Category')
ax.set_xlabel('Seller Tier')
ax.set_ylabel('Number of Sellers')
annotate_bars(ax)
plt.tight_layout()
plt.show()

tier_palette = {'Good': '#55A868', 'Average': '#F0C808', 'Poor': '#C44E52'}
ax = sns.scatterplot(data=seller_score, x='cancellation_rate', y='avg_review_score', hue='performance_tier', palette=tier_palette, alpha=0.8)
ax.set_title('Seller Cancellation Rate vs Average Review Score')
ax.set_xlabel('Cancellation Rate %')
ax.set_ylabel('Average Review Score')
for _, row in seller_score.sort_values('cancellation_rate', ascending=False).head(5).iterrows():
    ax.annotate(row['seller_id'][:6], (row['cancellation_rate'], row['avg_review_score']), fontsize=8)
plt.tight_layout()
plt.show()

seller_score[seller_score['risk_flag'] == 'At Risk'].sort_values('total_revenue').head(10)

**Seller management recommendation:** Poor sellers need targeted review because high cancellations, weak reviews, or late delivery can damage customer experience. At Risk sellers should be monitored for low revenue contribution and operational value.

## Section 10 - Business Recommendations

1. Problem: Some categories may generate high revenue but weak estimated margin after freight. Finding: [X%] of analyzed categories show estimated profit margin below the target threshold. Recommendation: Review pricing, packaging, and freight contracts for low-margin categories before increasing marketing spend.

2. Problem: Freight cost can reduce profitability even when sales volume is healthy. Finding: [X] categories are flagged as Freight Trap categories. Recommendation: Renegotiate shipping rates or introduce category-specific freight thresholds for these items.

3. Problem: Canceled and unavailable orders represent measurable revenue leakage. Finding: Cancellation loss is concentrated in [X] product categories. Recommendation: Improve inventory availability and seller fulfillment checks for high-loss categories.

4. Problem: Delivery delays can weaken customer experience and review scores. Finding: [X] states show the highest average delivery delays. Recommendation: Prioritize logistics improvement and carrier review in delayed states.

5. Problem: Seller performance is uneven across cancellation, review, and delivery metrics. Finding: [X%] of sellers fall into the Poor performance tier. Recommendation: Create seller coaching, SLA tracking, and escalation actions for Poor sellers.

6. Problem: Low-revenue sellers may add operational complexity without strong contribution. Finding: Bottom 10% sellers contribute [X] revenue and are flagged At Risk. Recommendation: Monitor At Risk sellers and decide whether to support, consolidate, or remove inactive sellers.

7. Problem: Payment and order status patterns may reveal operational friction. Finding: [X] payment methods and order statuses dominate transaction behavior. Recommendation: Use payment and status trends to improve checkout experience and reduce failed fulfillment.